# Gold layer - route KPI daily

Aggregate route-level window KPIs into daily summaries.
This step produces a route-by-date Gold table for downstream reporting and trend analysis.

In [0]:
_ = spark.sql("USE azure_streaming_mvp")

### Daily KPI construction

Aggregate `gold_route_kpi_window` into daily route-level summaries.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_route_kpi_daily
USING DELTA
AS
SELECT
  DATE(window_start) AS date,
  route_id,

  -- Weighted average of delay to account for different event volumes per window
  SUM(avg_delay_sec * n_events_delay) / NULLIF(SUM(n_events_delay), 0) AS avg_delay_sec,
  AVG(avg_occupancy_pct) AS avg_occupancy_pct,

  SUM(COALESCE(n_events_delay, 0)) AS total_events_delay,
  SUM(COALESCE(n_events_occupancy, 0)) AS total_events_occupancy,

  AVG(COALESCE(late_rate_delay, 0.0)) AS avg_late_rate_delay,
  AVG(COALESCE(avg_ingest_delay_sec, 0.0)) AS avg_ingest_delay_sec,

  CASE
    WHEN SUM(CASE WHEN dq_flag <> 'OK' THEN 1 ELSE 0 END) > 0 THEN 'CHECK'
    ELSE 'OK'
  END AS dq_flag

FROM gold_route_kpi_window
GROUP BY DATE(window_start), route_id;

num_affected_rows,num_inserted_rows


## Data validation checks

These checks confirm that the daily Gold table was built successfully
and summarize the current data-quality status.

In [0]:
%sql
SELECT metric, value
FROM (

  SELECT
    1 AS sort_order,
    'row_count' AS metric,
    CAST(COUNT(*) AS STRING) AS value
  FROM gold_route_kpi_daily

  UNION ALL

  SELECT
    2 AS sort_order,
    'latest_date' AS metric,
    CAST(MAX(date) AS STRING) AS value
  FROM gold_route_kpi_daily

  UNION ALL

  SELECT
    3 AS sort_order,
    CONCAT('dq_flag=', dq_flag) AS metric,
    CAST(MAX(date) AS STRING) AS value
  FROM gold_route_kpi_daily
  GROUP BY dq_flag
)
ORDER BY sort_order, metric;

metric,value
row_count,8
latest_date,2026-03-09
dq_flag=CHECK,2026-03-09


### Preview daily route KPIs

Inspect recent daily KPI records from the Gold layer.

In [0]:
%sql
-- Preview daily KPIs
SELECT *
FROM gold_route_kpi_daily
ORDER BY date DESC, route_id
LIMIT 50;

date,route_id,avg_delay_sec,avg_occupancy_pct,total_events_delay,total_events_occupancy,avg_late_rate_delay,avg_ingest_delay_sec,dq_flag
2026-03-09,B1,219.0,29.181818181818183,18,16,0.7692307692307693,1629.7948717948716,CHECK
2026-03-09,B2,388.0,44.03125,12,16,0.6363636363636364,1216.0,CHECK
2026-03-09,M1,226.42857142857142,58.7,14,6,0.9,1669.97,CHECK
2026-03-09,M2,290.0,26.869047619047617,7,16,0.4444444444444444,931.1111111111111,CHECK
2026-03-09,R10,241.36363636363637,47.1875,11,16,0.75,1139.5625,CHECK
2026-03-09,T1,291.7142857142857,52.6875,14,9,0.75,1347.388888888889,CHECK
2026-03-09,X3,293.2857142857143,30.25,14,10,0.9,1573.55,CHECK
2026-03-09,X7,281.9166666666667,33.666666666666664,12,9,0.7272727272727273,1263.5,CHECK


In [0]:
# ---- Notebook completion signal ----
dbutils.notebook.exit("OK")